# FRED Data Preprocessing
Cleaning the raw risk-free rate and CPI series: filling gaps, converting units, and aligning to trading dates.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW_DATA_PATH = Path('../data/raw')
PROCESSED_DATA_PATH = Path('../data/processed')

## Load raw data

In [3]:
rf = pd.read_csv(RAW_DATA_PATH / 'fred_risk_free_rate_raw.csv', index_col='date', parse_dates=True)
cpi = pd.read_csv(RAW_DATA_PATH / 'fred_cpi_raw.csv', index_col='date', parse_dates=True)

print("Risk-free rate shape:", rf.shape)
print("CPI shape:", cpi.shape)

Risk-free rate shape: (11696, 1)
CPI shape: (953, 1)


## Clean the risk-free rate series
FRED's Treasury series has gaps on weekends/holidays. We reindex to every calendar day and forward-fill,
since a rate stays constant until the next published update.

In [4]:
rf_filled = rf.asfreq('D').ffill()

print("Missing values after ffill:", rf_filled['risk_free_rate_pct'].isna().sum())
rf_filled.head(10)

Missing values after ffill: 0


,risk_free_rate_pct
date,
1981-09-01,17.01
1981-09-02,16.65
1981-09-03,16.96
1981-09-04,16.64
1981-09-05,16.64
1981-09-06,16.64
1981-09-07,16.64
1981-09-08,16.54
1981-09-09,16.39


## Convert annualized percentage to decimal
FRED reports this as an annualized percent (e.g. 5.25 = 5.25%). Convert to decimal for use in Sharpe ratio math.

In [5]:
rf_filled['risk_free_rate_decimal'] = rf_filled['risk_free_rate_pct'] / 100
rf_filled.head()

,risk_free_rate_pct,risk_free_rate_decimal
date,,
1981-09-01,17.01,0.1701
1981-09-02,16.65,0.1665
1981-09-03,16.96,0.1696
1981-09-04,16.64,0.1664
1981-09-05,16.64,0.1664


## Align to trading dates
Replace this with the actual trading date index from the price data notebook once available.
For now this uses US business days as a placeholder.

In [6]:
# TODO: replace with actual trading dates from price data, e.g.:
# trading_dates = pd.read_csv(PROCESSED_DATA_PATH / 'prices_processed.csv', index_col='date', parse_dates=True).index

trading_dates = pd.bdate_range(start=rf_filled.index.min(), end=rf_filled.index.max())
rf_aligned = rf_filled.reindex(trading_dates).ffill()
rf_aligned.index.name = 'date'

print(f"Aligned to {len(rf_aligned)} trading days")
rf_aligned.head()

Aligned to 11696 trading days


,risk_free_rate_pct,risk_free_rate_decimal
date,,
1981-09-01,17.01,0.1701
1981-09-02,16.65,0.1665
1981-09-03,16.96,0.1696
1981-09-04,16.64,0.1664
1981-09-07,16.64,0.1664


## Clean CPI (monthly -> forward-filled to daily, for merging with daily context if needed)

In [7]:
cpi_filled = cpi.asfreq('D').ffill()
cpi_filled['cpi_pct_change'] = cpi_filled['cpi_index'].pct_change()
cpi_filled.head()

,cpi_index,cpi_pct_change
date,,
1947-01-01,21.48,NaN
1947-01-02,21.48,0.0
1947-01-03,21.48,0.0
1947-01-04,21.48,0.0
1947-01-05,21.48,0.0


## Save processed data

In [8]:
rf_aligned.to_csv(PROCESSED_DATA_PATH / 'fred_risk_free_rate_processed.csv')
cpi_filled.to_csv(PROCESSED_DATA_PATH / 'fred_cpi_processed.csv')

print("Saved processed FRED data to", PROCESSED_DATA_PATH)

Saved processed FRED data to ../data/processed
